# Data Pipeline

This notebook handles the complete daily data pipeline with safety checks:

## Features
1. **Dual Inbound Support** - Processes both depot-level and service-level files
2. **Date Validation** - Filename-to-data matching, future date rejection, gap detection
3. **RAW Layer Management** - Upsert into RAW masters (depot & service)
4. **GOLD Layer Building** - Merge with calendars to create enriched data
5. **Predictions Update** - Update prediction tracker with actuals
6. **Logging & Archiving** - Track all ingestions and errors

## File Types
| File Pattern | Type | Description |
|--------------|------|-------------|
| `ops_daily_YYYY-MM-DD.csv` | Depot-level | Aggregated daily metrics per depot |
| `ops_daily_service_YYYY-MM-DD.csv` | Service-level | Detailed metrics per service |

## Output Structure
```
data/
├── raw/
│   ├── ops_daily_master.csv           # Depot-level RAW
│   └── ops_daily_service_master.csv   # Service-level RAW
├── processed/
│   ├── ops_daily_gold.parquet         # Depot-level GOLD
│   └── ops_daily_service_gold.parquet # Service-level GOLD
├── archive/inbound_daily/             # Processed files
└── logs/
    ├── ingest_log.csv                 # Successful ingestions
    └── ingest_errors.csv              # Failed ingestions
```

## 1. Setup & Configuration

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import hashlib
import re
from datetime import datetime, date, timedelta
import json
import warnings
from typing import Optional, Tuple, List, Dict

warnings.filterwarnings('ignore')

print("Imports loaded successfully")

Imports loaded successfully


In [2]:
# =============================================================================
# PROJECT PATHS
# =============================================================================

# Project root - find dynamically by looking for marker directories
def find_project_root():
    """Find project root by looking for data/ or model/ directory."""
    # Start from notebook location (assuming notebooks/ folder)
    candidates = [
        Path("..").resolve(),           # If running from notebooks/
        Path(".").resolve(),            # If running from project root
        Path(__file__).parent.parent if '__file__' in dir() else None,
    ]
    
    for candidate in candidates:
        if candidate and (candidate / "data").exists() and (candidate / "model").exists():
            return candidate
    
    # Fallback: explicit path
    fallback = Path("/home/sr/agnigent/ai_agents/dynamic_scheduling")
    if fallback.exists():
        return fallback
    
    raise RuntimeError("Could not find project root. Expected data/ and model/ directories.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

# Input/Output directories
INBOUND_DIR = DATA_DIR / "inbound_daily"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVE_DIR = DATA_DIR / "archive" / "inbound_daily"
LOGS_DIR = DATA_DIR / "logs"
PREDICTIONS_DIR = PROJECT_ROOT / "output" / "predictions"

# Depot-level master files
RAW_MASTER_CSV = RAW_DIR / "ops_daily_master.csv"
GOLD_MASTER_PARQ = PROCESSED_DIR / "ops_daily_gold.parquet"

# Service-level master files
RAW_SERVICE_MASTER_CSV = RAW_DIR / "ops_daily_service_master.csv"
GOLD_SERVICE_PARQ = PROCESSED_DIR / "ops_daily_service_gold.parquet"

# Log files
INGEST_LOG_CSV = LOGS_DIR / "ingest_log.csv"
ERROR_LOG_CSV = LOGS_DIR / "ingest_errors.csv"

# Predictions file
PREDICTIONS_FILE = PREDICTIONS_DIR / "daily_predictions.parquet"

# Calendar files (master data)
TELUGU_CAL_PATH = DATA_DIR / "master" / "telugu_calendar.csv"
HOLIDAY_CAL_PATH = DATA_DIR / "master" / "holiday_calendar.csv"

# Create directories if they don't exist
for d in [INBOUND_DIR, RAW_DIR, PROCESSED_DIR, ARCHIVE_DIR, LOGS_DIR, PREDICTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT        :", PROJECT_ROOT)
print("\nInbound/Output:")
print("  INBOUND_DIR       :", INBOUND_DIR)
print("  RAW_MASTER        :", RAW_MASTER_CSV, f"(exists: {RAW_MASTER_CSV.exists()})")
print("  RAW_SERVICE_MASTER:", RAW_SERVICE_MASTER_CSV, f"(exists: {RAW_SERVICE_MASTER_CSV.exists()})")
print("  GOLD_PARQUET      :", GOLD_MASTER_PARQ, f"(exists: {GOLD_MASTER_PARQ.exists()})")
print("  GOLD_SERVICE_PARQ :", GOLD_SERVICE_PARQ, f"(exists: {GOLD_SERVICE_PARQ.exists()})")
print("\nCalendars:")
print("  TELUGU_CAL        :", TELUGU_CAL_PATH, f"(exists: {TELUGU_CAL_PATH.exists()})")
print("  HOLIDAY_CAL       :", HOLIDAY_CAL_PATH, f"(exists: {HOLIDAY_CAL_PATH.exists()})")

PROJECT_ROOT        : /home/sr/agnigent/ai_agents/dynamic_scheduling

Inbound/Output:
  INBOUND_DIR       : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/inbound_daily
  RAW_MASTER        : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/raw/ops_daily_master.csv (exists: True)
  RAW_SERVICE_MASTER: /home/sr/agnigent/ai_agents/dynamic_scheduling/data/raw/ops_daily_service_master.csv (exists: True)
  GOLD_PARQUET      : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/processed/ops_daily_gold.parquet (exists: True)
  GOLD_SERVICE_PARQ : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/processed/ops_daily_service_gold.parquet (exists: False)

Calendars:
  TELUGU_CAL        : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/master/telugu_calendar.csv (exists: True)
  HOLIDAY_CAL       : /home/sr/agnigent/ai_agents/dynamic_scheduling/data/master/holiday_calendar.csv (exists: True)


In [3]:
# =============================================================================
# SCHEMA CONFIGURATION
# =============================================================================

# Allowed depots (update as you scale)
ALLOWED_DEPOTS = ["CONTONMENT", "KARIMNAGAR-I", "NIZAMABAD-I", "WARANGAL-I"]

# --- Depot-level schema ---
INBOUND_COLUMNS = ["depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio"]
RAW_COLUMNS = INBOUND_COLUMNS.copy()

# --- Service-level schema ---
SERVICE_INBOUND_COLUMNS = [
    "depot", "date", "service_number", 
    "actual_kms", "actual_trips", "seat_kms", "passenger_kms", "occupancy_ratio"
]
SERVICE_RAW_COLUMNS = SERVICE_INBOUND_COLUMNS.copy()

# --- Calendar schemas ---
TELUGU_CAL_COLUMNS = ["date", "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day"]
HOLIDAY_WIDE_COLUMNS = ["fes_hol_code", "Holiday_Festival", "fes_hol_category",
                        "2023_dates", "2024_dates", "2025_dates", "2026_dates"]

# --- GOLD schemas ---
GOLD_COLUMNS = [
    "depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio",
    "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day",
    "fes_hol_code", "Holiday_Festival", "fes_hol_category", "is_fes_hol"
]

SERVICE_GOLD_COLUMNS = [
    "depot", "date", "service_number",
    "actual_kms", "actual_trips", "seat_kms", "passenger_kms", "occupancy_ratio",
    "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day",
    "fes_hol_code", "Holiday_Festival", "fes_hol_category", "is_fes_hol"
]

# --- Validation bounds ---
BOUNDS = {
    "passengers_per_day": (0, None),
    "actual_kms": (0, None),
    "occupancy_ratio": (0, 2.0),
}

SERVICE_BOUNDS = {
    "actual_kms": (0, None),
    "actual_trips": (0, None),
    "seat_kms": (0, None),
    "passenger_kms": (0, None),
    "occupancy_ratio": (0, 3.0),  # Service-level can be higher due to standees
}

# --- File patterns ---
DEPOT_FILE_PATTERN = re.compile(r"ops_daily_(\d{4}-\d{2}-\d{2})\.csv$")
SERVICE_FILE_PATTERN = re.compile(r"ops_daily_service_(\d{4}-\d{2}-\d{2})\.csv$")

print("Schema Configuration:")
print(f"  Allowed depots: {ALLOWED_DEPOTS}")
print(f"  Depot-level columns: {len(INBOUND_COLUMNS)}")
print(f"  Service-level columns: {len(SERVICE_INBOUND_COLUMNS)}")

Schema Configuration:
  Allowed depots: ['CONTONMENT', 'KARIMNAGAR-I', 'NIZAMABAD-I', 'WARANGAL-I']
  Depot-level columns: 5
  Service-level columns: 8


## 2. Helper Functions

In [4]:
# =============================================================================
# HELPER FUNCTIONS - File I/O
# =============================================================================

def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    """Write CSV atomically to prevent corruption."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def atomic_write_parquet(df: pd.DataFrame, path: Path) -> None:
    """Write Parquet atomically to prevent corruption."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, engine="pyarrow", index=False)
    tmp.replace(path)


def file_sha256(path: Path) -> str:
    """Calculate SHA256 hash of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def normalize_date_key(s: pd.Series) -> pd.Series:
    """Convert date series to normalized Timestamp at midnight."""
    return pd.to_datetime(s, errors="coerce").dt.normalize()


print("Helper functions defined")

Helper functions defined


## 3. Date Validation & Safety Checks

In [5]:
# =============================================================================
# DATE VALIDATION FUNCTIONS
# =============================================================================

def extract_date_from_filename(filename: str, file_type: str = "depot") -> Optional[date]:
    """
    Extract date from filename.
    
    Expected patterns:
        depot: ops_daily_YYYY-MM-DD.csv
        service: ops_daily_service_YYYY-MM-DD.csv
    """
    pattern = DEPOT_FILE_PATTERN if file_type == "depot" else SERVICE_FILE_PATTERN
    match = pattern.search(filename)
    
    if match:
        date_str = match.group(1)
        try:
            return datetime.strptime(date_str, "%Y-%m-%d").date()
        except ValueError:
            return None
    return None


def validate_filename_format(filepath: Path) -> Tuple[bool, str, Optional[str]]:
    """
    Validate filename format and determine file type.
    
    Returns:
        (is_valid, file_type, error_message)
        file_type: 'depot' or 'service'
    """
    filename = filepath.name
    
    if DEPOT_FILE_PATTERN.search(filename):
        return True, "depot", None
    elif SERVICE_FILE_PATTERN.search(filename):
        return True, "service", None
    else:
        return False, "unknown", (
            f"Invalid filename format: {filename}. "
            f"Expected: ops_daily_YYYY-MM-DD.csv or ops_daily_service_YYYY-MM-DD.csv"
        )


def validate_date_consistency(
    filepath: Path,
    data_date: date,
    file_type: str
) -> Tuple[bool, Optional[str]]:
    """
    Validate that filename date matches data date.
    
    Returns:
        (is_valid, error_message)
    """
    filename_date = extract_date_from_filename(filepath.name, file_type)
    
    if filename_date is None:
        return False, f"Could not extract date from filename: {filepath.name}"
    
    if filename_date != data_date:
        return False, (
            f"Date mismatch! Filename date: {filename_date}, "
            f"Data date: {data_date}. File: {filepath.name}"
        )
    
    return True, None


def validate_not_future_date(data_date: date) -> Tuple[bool, Optional[str]]:
    """
    Validate that date is not in the future.
    
    Returns:
        (is_valid, error_message)
    """
    today = date.today()
    
    if data_date > today:
        return False, f"Future date not allowed: {data_date} (today is {today})"
    
    return True, None


def check_already_processed(
    data_date: date,
    master_path: Path,
    date_col: str = "date"
) -> Tuple[bool, int]:
    """
    Check if date already exists in master file.
    
    Returns:
        (already_exists, existing_row_count)
    """
    if not master_path.exists():
        return False, 0
    
    try:
        df = pd.read_csv(master_path)
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        
        existing = df[df[date_col].dt.date == data_date]
        return len(existing) > 0, len(existing)
    except Exception:
        return False, 0


def detect_date_gaps(
    data_date: date,
    master_path: Path,
    date_col: str = "date",
    max_gap_days: int = 30
) -> List[date]:
    """
    Detect missing dates between last processed date and new date.
    
    Returns:
        List of missing dates
    """
    if not master_path.exists():
        return []
    
    try:
        df = pd.read_csv(master_path)
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        
        if len(df) == 0:
            return []
        
        last_date = df[date_col].max().date()
        
        if data_date <= last_date:
            return []  # Re-processing or same date
        
        # Find gaps
        existing_dates = set(df[date_col].dt.date.unique())
        missing = []
        
        current = last_date + timedelta(days=1)
        while current < data_date and len(missing) < max_gap_days:
            if current not in existing_dates:
                missing.append(current)
            current += timedelta(days=1)
        
        return missing
    except Exception:
        return []


print("Date validation functions defined:")
print("  - extract_date_from_filename()")
print("  - validate_filename_format()")
print("  - validate_date_consistency()")
print("  - validate_not_future_date()")
print("  - check_already_processed()")
print("  - detect_date_gaps()")

Date validation functions defined:
  - extract_date_from_filename()
  - validate_filename_format()
  - validate_date_consistency()
  - validate_not_future_date()
  - check_already_processed()
  - detect_date_gaps()


## 4. Error Logging

In [6]:
# =============================================================================
# ERROR LOGGING FUNCTIONS
# =============================================================================

def log_error(
    filepath: Path,
    error_type: str,
    error_message: str,
    step_failed: str = "unknown"
) -> None:
    """
    Log an error to the error log file.
    """
    error_record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "file": str(filepath.name) if filepath else "N/A",
        "file_path": str(filepath) if filepath else "N/A",
        "error_type": error_type,
        "error_message": error_message,
        "step_failed": step_failed,
    }
    
    error_df = pd.DataFrame([error_record])
    
    if ERROR_LOG_CSV.exists():
        existing = pd.read_csv(ERROR_LOG_CSV)
        combined = pd.concat([existing, error_df], ignore_index=True)
        atomic_write_csv(combined, ERROR_LOG_CSV)
    else:
        atomic_write_csv(error_df, ERROR_LOG_CSV)
    
    print(f"  ERROR logged: {error_type} - {error_message[:50]}...")


def log_success(
    record: dict
) -> None:
    """
    Log a successful ingestion to the ingest log.
    """
    log_df = pd.DataFrame([record])
    
    if INGEST_LOG_CSV.exists():
        existing = pd.read_csv(INGEST_LOG_CSV)
        combined = pd.concat([existing, log_df], ignore_index=True)
        atomic_write_csv(combined, INGEST_LOG_CSV)
    else:
        atomic_write_csv(log_df, INGEST_LOG_CSV)


print("Error logging functions defined:")
print("  - log_error()")
print("  - log_success()")

Error logging functions defined:
  - log_error()
  - log_success()


## 5. Depot-Level Inbound Processing

In [7]:
# =============================================================================
# DEPOT-LEVEL INBOUND FUNCTIONS
# =============================================================================

def read_inbound_csv(inbound_path: Path) -> pd.DataFrame:
    """Read inbound CSV file, removing any unnamed columns."""
    df = pd.read_csv(inbound_path)
    df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    return df


def coerce_depot_types(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce depot-level inbound data to correct types."""
    df = df.copy()
    
    # Parse date with dayfirst=True (DD-MM-YYYY)
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="raise")
    
    # Strings
    df["depot"] = df["depot"].astype("string").str.strip()
    
    # Numerics
    for col in ["passengers_per_day", "actual_kms", "occupancy_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    return df


def validate_depot_inbound(df: pd.DataFrame) -> List[str]:
    """Validate depot-level inbound data. Returns list of error messages."""
    errors = []
    
    # Check required columns
    missing = [c for c in INBOUND_COLUMNS if c not in df.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")
        return errors
    
    # Check for null keys
    if df["depot"].isna().any():
        errors.append("Null depot found.")
    if df["date"].isna().any():
        errors.append("Null/invalid date found.")
    
    # One date per file
    if df["date"].nunique() != 1:
        errors.append(f"File must have exactly one date; found {df['date'].nunique()}")
    
    # Unique depot-date
    if df.duplicated(subset=["depot", "date"]).any():
        errors.append("Duplicate (depot, date) rows found.")
    
    # Depot whitelist
    unknown = sorted(set(df["depot"].dropna().astype(str)) - set(ALLOWED_DEPOTS))
    if unknown:
        errors.append(f"Unknown depots: {unknown}")
    
    # Expected row count
    if len(df) != len(ALLOWED_DEPOTS):
        errors.append(f"Row count {len(df)} but expected {len(ALLOWED_DEPOTS)} (one per depot).")
    
    # Sanity bounds
    for col, (lo, hi) in BOUNDS.items():
        if col not in df.columns:
            continue
        if lo is not None and (df[col] < lo).any():
            errors.append(f"{col} has values < {lo}")
        if hi is not None and (df[col] > hi).any():
            errors.append(f"{col} has values > {hi}")
    
    return errors


print("Depot-level inbound functions defined")

Depot-level inbound functions defined


In [8]:
# =============================================================================
# DEPOT-LEVEL RAW MASTER FUNCTIONS
# =============================================================================

def load_depot_raw_master(path: Path) -> pd.DataFrame:
    """Load depot-level RAW master or create empty DataFrame."""
    if not path.exists():
        return pd.DataFrame(columns=RAW_COLUMNS)
    
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df["depot"] = df["depot"].astype("string").str.strip()
    
    for col in ["passengers_per_day", "actual_kms", "occupancy_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    df = df[RAW_COLUMNS].sort_values(["depot", "date"]).reset_index(drop=True)
    return df


def upsert_depot_raw_master(
    master: pd.DataFrame, 
    inbound: pd.DataFrame
) -> Tuple[pd.DataFrame, int, int]:
    """
    Upsert depot-level data into RAW master.
    Returns: (updated_master, inserted_count, corrected_count)
    """
    master = master.copy()
    inbound = inbound.copy()[RAW_COLUMNS]
    
    master_keys = set(zip(master["depot"].astype(str), master["date"].dt.date))
    inbound_keys = list(zip(inbound["depot"].astype(str), inbound["date"].dt.date))
    
    existing = [k in master_keys for k in inbound_keys]
    corrected = int(np.sum(existing))
    inserted = int(len(inbound) - corrected)
    
    # Remove matching keys, then append inbound
    key_df = inbound[["depot", "date"]].copy()
    merged = master.merge(key_df, on=["depot", "date"], how="left", indicator=True)
    master_kept = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
    
    updated = pd.concat([master_kept, inbound], ignore_index=True)
    updated = updated.sort_values(["depot", "date"]).reset_index(drop=True)
    
    if updated.duplicated(subset=["depot", "date"]).any():
        raise ValueError("RAW master has duplicate (depot, date) after upsert.")
    
    return updated, inserted, corrected


print("Depot-level RAW master functions defined")

Depot-level RAW master functions defined


## 6. Service-Level Inbound Processing

In [9]:
# =============================================================================
# SERVICE-LEVEL INBOUND FUNCTIONS
# =============================================================================

def coerce_service_types(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce service-level inbound data to correct types."""
    df = df.copy()
    
    # Parse date with dayfirst=True
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="raise")
    
    # Strings
    df["depot"] = df["depot"].astype("string").str.strip()
    df["service_number"] = df["service_number"].astype("string").str.strip()
    
    # Numerics
    for col in ["actual_kms", "actual_trips", "seat_kms", "passenger_kms", "occupancy_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    return df


def validate_service_inbound(df: pd.DataFrame) -> List[str]:
    """Validate service-level inbound data. Returns list of error messages."""
    errors = []
    
    # Check required columns
    missing = [c for c in SERVICE_INBOUND_COLUMNS if c not in df.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")
        return errors
    
    # Check for null keys
    if df["depot"].isna().any():
        errors.append("Null depot found.")
    if df["date"].isna().any():
        errors.append("Null/invalid date found.")
    if df["service_number"].isna().any():
        errors.append("Null service_number found.")
    
    # One date per file
    if df["date"].nunique() != 1:
        errors.append(f"File must have exactly one date; found {df['date'].nunique()}")
    
    # Unique depot-date-service
    if df.duplicated(subset=["depot", "date", "service_number"]).any():
        errors.append("Duplicate (depot, date, service_number) rows found.")
    
    # Depot whitelist
    unknown = sorted(set(df["depot"].dropna().astype(str)) - set(ALLOWED_DEPOTS))
    if unknown:
        errors.append(f"Unknown depots: {unknown}")
    
    # Sanity bounds
    for col, (lo, hi) in SERVICE_BOUNDS.items():
        if col not in df.columns:
            continue
        if lo is not None and (df[col] < lo).any():
            errors.append(f"{col} has values < {lo}")
        if hi is not None and (df[col] > hi).any():
            errors.append(f"{col} has values > {hi}")
    
    return errors


print("Service-level inbound functions defined")

Service-level inbound functions defined


In [10]:
# =============================================================================
# SERVICE-LEVEL RAW MASTER FUNCTIONS
# =============================================================================

def load_service_raw_master(path: Path) -> pd.DataFrame:
    """Load service-level RAW master or create empty DataFrame."""
    if not path.exists():
        return pd.DataFrame(columns=SERVICE_RAW_COLUMNS)
    
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df["depot"] = df["depot"].astype("string").str.strip()
    df["service_number"] = df["service_number"].astype("string").str.strip()
    
    for col in ["actual_kms", "actual_trips", "seat_kms", "passenger_kms", "occupancy_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Ensure all columns exist
    for col in SERVICE_RAW_COLUMNS:
        if col not in df.columns:
            df[col] = None
    
    df = df[SERVICE_RAW_COLUMNS].sort_values(["depot", "date", "service_number"]).reset_index(drop=True)
    return df


def upsert_service_raw_master(
    master: pd.DataFrame, 
    inbound: pd.DataFrame
) -> Tuple[pd.DataFrame, int, int]:
    """
    Upsert service-level data into RAW master.
    Returns: (updated_master, inserted_count, corrected_count)
    """
    master = master.copy()
    inbound = inbound.copy()
    
    # Ensure columns
    for col in SERVICE_RAW_COLUMNS:
        if col not in inbound.columns:
            inbound[col] = None
    inbound = inbound[SERVICE_RAW_COLUMNS]
    
    # Create composite key
    master_keys = set(zip(
        master["depot"].astype(str), 
        master["date"].dt.date,
        master["service_number"].astype(str)
    ))
    inbound_keys = list(zip(
        inbound["depot"].astype(str), 
        inbound["date"].dt.date,
        inbound["service_number"].astype(str)
    ))
    
    existing = [k in master_keys for k in inbound_keys]
    corrected = int(np.sum(existing))
    inserted = int(len(inbound) - corrected)
    
    # Remove matching keys, then append inbound
    key_df = inbound[["depot", "date", "service_number"]].copy()
    merged = master.merge(key_df, on=["depot", "date", "service_number"], how="left", indicator=True)
    master_kept = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
    
    updated = pd.concat([master_kept, inbound], ignore_index=True)
    updated = updated.sort_values(["depot", "date", "service_number"]).reset_index(drop=True)
    
    if updated.duplicated(subset=["depot", "date", "service_number"]).any():
        raise ValueError("Service RAW master has duplicate keys after upsert.")
    
    return updated, inserted, corrected


print("Service-level RAW master functions defined")

Service-level RAW master functions defined


## 7. Calendar Loading

In [11]:
# =============================================================================
# CALENDAR LOADING FUNCTIONS
# =============================================================================

def load_telugu_calendar(path: Path) -> pd.DataFrame:
    """Load and parse Telugu calendar CSV."""
    if not path.exists():
        raise FileNotFoundError(f"Missing telugu_calendar.csv at: {path}")
    
    cal = pd.read_csv(path)
    cal = cal.loc[:, ~cal.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    
    missing = [c for c in TELUGU_CAL_COLUMNS if c not in cal.columns]
    if missing:
        raise ValueError(f"Telugu calendar missing columns: {missing}")
    
    cal["date"] = cal["date"].astype(str).str.strip()
    cal["date"] = cal["date"].str.replace(r"\s+00:00:00$", "", regex=True)
    cal["date"] = pd.to_datetime(cal["date"], format="mixed", dayfirst=True, errors="coerce").dt.normalize()
    
    if cal["date"].isna().any():
        bad = cal[cal["date"].isna()][["date"]].head(10)
        raise ValueError(f"Unparseable dates in telugu_calendar.csv. Examples:\n{bad}")
    
    cal["telugu_thithi"] = cal["telugu_thithi"].astype("string").str.strip()
    cal["telugu_paksham"] = cal["telugu_paksham"].astype("string").str.strip()
    cal["telugu_month"] = cal["telugu_month"].astype("string").str.strip()
    cal["marriage_day"] = pd.to_numeric(cal["marriage_day"], errors="coerce").fillna(0.0)
    cal["moudyami_day"] = pd.to_numeric(cal["moudyami_day"], errors="coerce").fillna(0.0)
    
    cal = cal[TELUGU_CAL_COLUMNS].sort_values("date").drop_duplicates(subset=["date"])
    return cal


def _split_date_cell(cell) -> list:
    """Parse holiday calendar cells with multiple dates."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    s = s.replace("\n", ",").replace(";", ",").replace("|", ",")
    return [p.strip() for p in s.split(",") if p.strip()]


def load_holiday_calendar_long(path: Path) -> pd.DataFrame:
    """Load holiday calendar and convert from wide to long format."""
    if not path.exists():
        raise FileNotFoundError(f"Missing holiday_calendar.csv at: {path}")
    
    hol = pd.read_csv(path)
    hol = hol.loc[:, ~hol.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    
    missing = [c for c in HOLIDAY_WIDE_COLUMNS if c not in hol.columns]
    if missing:
        raise ValueError(f"Holiday calendar missing columns: {missing}")
    
    year_cols = ["2023_dates", "2024_dates", "2025_dates", "2026_dates"]
    long = hol.melt(
        id_vars=["fes_hol_code", "Holiday_Festival", "fes_hol_category"],
        value_vars=year_cols,
        var_name="year_col",
        value_name="date_list",
    )
    
    long["date_strs"] = long["date_list"].apply(_split_date_cell)
    long = long.explode("date_strs", ignore_index=True)
    long["date"] = pd.to_datetime(long["date_strs"], dayfirst=True, errors="coerce")
    long = long.dropna(subset=["date"])
    
    long = long[["date", "fes_hol_code", "Holiday_Festival", "fes_hol_category"]].copy()
    long["fes_hol_code"] = pd.to_numeric(long["fes_hol_code"], errors="coerce").astype("Int64")
    long = long.sort_values(["date", "fes_hol_code"]).drop_duplicates(subset=["date"], keep="first").reset_index(drop=True)
    
    return long


print("Calendar loading functions defined")

Calendar loading functions defined


## 8. GOLD Layer Building

In [12]:
# =============================================================================
# GOLD LAYER BUILDING - DEPOT LEVEL
# =============================================================================

def build_depot_gold(
    raw_df: pd.DataFrame,
    telugu_cal_df: pd.DataFrame,
    holiday_long_df: pd.DataFrame
) -> pd.DataFrame:
    """Build depot-level GOLD layer by merging RAW with calendars."""
    df = raw_df.copy()
    
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date_key"] = df["date"].dt.normalize()
    df["depot"] = df["depot"].astype("string")
    
    # Telugu calendar
    tel = telugu_cal_df.copy()
    tel["date"] = pd.to_datetime(tel["date"], errors="coerce")
    tel["date_key"] = tel["date"].dt.normalize()
    tel = tel.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    
    # Holiday calendar
    hol = holiday_long_df.copy()
    hol["date"] = pd.to_datetime(hol["date"], errors="coerce")
    hol["date_key"] = hol["date"].dt.normalize()
    hol = hol.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    
    # Merge
    df = df.merge(tel, on="date_key", how="left")
    df = df.merge(hol, on="date_key", how="left")
    
    # Holiday flag and defaults
    df["is_fes_hol"] = df["fes_hol_code"].notna()
    df["fes_hol_code"] = df["fes_hol_code"].fillna(0).astype("Int64")
    df["Holiday_Festival"] = df["Holiday_Festival"].fillna("NONE").astype("string")
    df["fes_hol_category"] = df["fes_hol_category"].fillna("NONE").astype("string")
    df["is_fes_hol"] = df["is_fes_hol"].astype("int8")
    
    df = df.drop(columns=["date_key"])
    df = df[GOLD_COLUMNS].sort_values(["depot", "date"]).reset_index(drop=True)
    
    if df.duplicated(subset=["depot", "date"]).any():
        raise ValueError("Depot GOLD has duplicate (depot, date).")
    
    return df


print("Depot-level GOLD building function defined")

Depot-level GOLD building function defined


In [13]:
# =============================================================================
# GOLD LAYER BUILDING - SERVICE LEVEL
# =============================================================================

def build_service_gold(
    raw_df: pd.DataFrame,
    telugu_cal_df: pd.DataFrame,
    holiday_long_df: pd.DataFrame
) -> pd.DataFrame:
    """Build service-level GOLD layer by merging RAW with calendars."""
    df = raw_df.copy()
    
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date_key"] = df["date"].dt.normalize()
    df["depot"] = df["depot"].astype("string")
    df["service_number"] = df["service_number"].astype("string")
    
    # Telugu calendar
    tel = telugu_cal_df.copy()
    tel["date"] = pd.to_datetime(tel["date"], errors="coerce")
    tel["date_key"] = tel["date"].dt.normalize()
    tel = tel.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    
    # Holiday calendar
    hol = holiday_long_df.copy()
    hol["date"] = pd.to_datetime(hol["date"], errors="coerce")
    hol["date_key"] = hol["date"].dt.normalize()
    hol = hol.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    
    # Merge
    df = df.merge(tel, on="date_key", how="left")
    df = df.merge(hol, on="date_key", how="left")
    
    # Holiday flag and defaults
    df["is_fes_hol"] = df["fes_hol_code"].notna()
    df["fes_hol_code"] = df["fes_hol_code"].fillna(0).astype("Int64")
    df["Holiday_Festival"] = df["Holiday_Festival"].fillna("NONE").astype("string")
    df["fes_hol_category"] = df["fes_hol_category"].fillna("NONE").astype("string")
    df["is_fes_hol"] = df["is_fes_hol"].astype("int8")
    
    df = df.drop(columns=["date_key"])
    
    # Ensure all columns exist
    for col in SERVICE_GOLD_COLUMNS:
        if col not in df.columns:
            df[col] = None
    
    df = df[SERVICE_GOLD_COLUMNS].sort_values(["depot", "date", "service_number"]).reset_index(drop=True)
    
    if df.duplicated(subset=["depot", "date", "service_number"]).any():
        raise ValueError("Service GOLD has duplicate (depot, date, service_number).")
    
    return df


print("Service-level GOLD building function defined")

Service-level GOLD building function defined


## 9. Predictions Tracker Integration

In [14]:
# =============================================================================
# PREDICTIONS TRACKER FUNCTIONS
# =============================================================================

PREDICTIONS_COLUMNS = [
    "run_date", "prediction_date", "depot",
    "predicted_passengers", "actual_passengers",
    "assumed_or", "actual_or", "estimated_kms", "actual_kms",
    "bus_capacity", "estimated_buses", "actual_buses",
    "passenger_error", "passenger_error_pct", "km_error", "km_error_pct", "status",
]


def load_predictions_file(file_path: Path) -> pd.DataFrame:
    """Load predictions file or create empty DataFrame."""
    if file_path.exists():
        df = pd.read_parquet(file_path)
        for col in PREDICTIONS_COLUMNS:
            if col not in df.columns:
                df[col] = None
        return df
    return pd.DataFrame(columns=PREDICTIONS_COLUMNS)


def save_predictions_file(df: pd.DataFrame, file_path: Path) -> None:
    """Save predictions DataFrame to parquet."""
    for col in ["run_date", "prediction_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    df.to_parquet(file_path, index=False)


def update_predictions_with_actuals(
    predictions_df: pd.DataFrame,
    gold_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, int]:
    """Update pending predictions with actuals. Returns (updated_df, count_updated)."""
    predictions_df = predictions_df.copy()
    
    if len(predictions_df) == 0:
        return predictions_df, 0
    
    predictions_df["prediction_date"] = pd.to_datetime(predictions_df["prediction_date"])
    gold_df = gold_df.copy()
    gold_df["date"] = pd.to_datetime(gold_df["date"])
    
    # Aggregate gold to depot-day
    agg_cols = {"passengers_per_day": "sum", "occupancy_ratio": "mean"}
    if "actual_kms" in gold_df.columns:
        agg_cols["actual_kms"] = "sum"
    
    actuals = gold_df.groupby(["depot", "date"]).agg(agg_cols).reset_index()
    
    pending_mask = predictions_df["status"] == "pending"
    updates_count = 0
    
    for pred_date in predictions_df.loc[pending_mask, "prediction_date"].unique():
        date_actuals = actuals[actuals["date"] == pred_date]
        if len(date_actuals) == 0:
            continue
        
        for depot in predictions_df.loc[
            (predictions_df["prediction_date"] == pred_date) & pending_mask, "depot"
        ].unique():
            depot_actual = date_actuals[date_actuals["depot"] == depot]
            if len(depot_actual) == 0:
                continue
            
            actual_passengers = depot_actual["passengers_per_day"].values[0]
            actual_or = depot_actual["occupancy_ratio"].values[0] if "occupancy_ratio" in depot_actual else None
            actual_kms = depot_actual["actual_kms"].values[0] if "actual_kms" in depot_actual.columns else None
            
            pred_mask = (
                (predictions_df["prediction_date"] == pred_date) &
                (predictions_df["depot"] == depot) &
                (predictions_df["status"] == "pending")
            )
            
            if not pred_mask.any():
                continue
            
            predicted = predictions_df.loc[pred_mask, "predicted_passengers"].values[0]
            estimated_kms = predictions_df.loc[pred_mask, "estimated_kms"].values[0]
            
            # Calculate errors
            passenger_error = predicted - actual_passengers if predicted else None
            passenger_error_pct = (passenger_error / actual_passengers * 100) if actual_passengers and actual_passengers > 0 else None
            
            km_error = None
            km_error_pct = None
            if actual_kms and actual_kms > 0 and estimated_kms:
                km_error = estimated_kms - actual_kms
                km_error_pct = (km_error / actual_kms * 100)
            
            # Update record
            predictions_df.loc[pred_mask, "actual_passengers"] = actual_passengers
            predictions_df.loc[pred_mask, "actual_or"] = actual_or
            predictions_df.loc[pred_mask, "actual_kms"] = actual_kms
            predictions_df.loc[pred_mask, "passenger_error"] = passenger_error
            predictions_df.loc[pred_mask, "passenger_error_pct"] = passenger_error_pct
            predictions_df.loc[pred_mask, "km_error"] = km_error
            predictions_df.loc[pred_mask, "km_error_pct"] = km_error_pct
            predictions_df.loc[pred_mask, "status"] = "completed"
            
            updates_count += 1
    
    return predictions_df, updates_count


print("Predictions tracker functions defined")

Predictions tracker functions defined


## 10. Archiving Functions

In [15]:
# =============================================================================
# ARCHIVING FUNCTIONS
# =============================================================================

def archive_inbound_file(inbound_path: Path, target_date: date) -> Path:
    """Move inbound file to archive directory."""
    date_str = target_date.strftime("%Y-%m-%d")
    archived_name = f"{inbound_path.stem}__archived__{date_str}{inbound_path.suffix}"
    archived_path = ARCHIVE_DIR / archived_name
    
    if archived_path.exists():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        archived_path = ARCHIVE_DIR / f"{inbound_path.stem}__archived__{date_str}__{ts}{inbound_path.suffix}"
    
    shutil.move(str(inbound_path), str(archived_path))
    return archived_path


print("Archiving functions defined")

Archiving functions defined


## 11. Pre-Flight Check & Summary

In [16]:
# =============================================================================
# PRE-FLIGHT CHECK FUNCTIONS
# =============================================================================

def scan_inbound_files() -> Dict[str, List[Path]]:
    """
    Scan inbound directory and categorize files.
    Returns: {"depot": [files], "service": [files], "unknown": [files]}
    """
    result = {"depot": [], "service": [], "unknown": []}
    
    for f in sorted(INBOUND_DIR.glob("*.csv")):
        is_valid, file_type, _ = validate_filename_format(f)
        if is_valid:
            result[file_type].append(f)
        else:
            result["unknown"].append(f)
    
    return result


def pre_flight_check(filepath: Path, file_type: str) -> Dict:
    """
    Run all validation checks on a file before processing.
    Returns dict with check results.
    """
    result = {
        "filepath": filepath,
        "file_type": file_type,
        "checks_passed": True,
        "errors": [],
        "warnings": [],
        "info": {},
    }
    
    try:
        # Read file
        df = read_inbound_csv(filepath)
        
        # Coerce types based on file type
        if file_type == "depot":
            df = coerce_depot_types(df)
            validation_errors = validate_depot_inbound(df)
            master_path = RAW_MASTER_CSV
        else:
            df = coerce_service_types(df)
            validation_errors = validate_service_inbound(df)
            master_path = RAW_SERVICE_MASTER_CSV
        
        # Get data date
        data_date = df["date"].iloc[0].date()
        result["info"]["data_date"] = data_date
        result["info"]["row_count"] = len(df)
        result["info"]["depots"] = df["depot"].unique().tolist()
        
        # Check 1: Filename-data date consistency
        date_valid, date_err = validate_date_consistency(filepath, data_date, file_type)
        if not date_valid:
            result["errors"].append(date_err)
            result["checks_passed"] = False
        
        # Check 2: Future date
        future_valid, future_err = validate_not_future_date(data_date)
        if not future_valid:
            result["errors"].append(future_err)
            result["checks_passed"] = False
        
        # Check 3: Schema validation
        if validation_errors:
            result["errors"].extend(validation_errors)
            result["checks_passed"] = False
        
        # Check 4: Already processed (warning only)
        already_exists, existing_count = check_already_processed(data_date, master_path)
        if already_exists:
            result["warnings"].append(
                f"Date {data_date} already exists in master ({existing_count} rows). Will be OVERWRITTEN."
            )
        result["info"]["already_processed"] = already_exists
        
        # Check 5: Date gaps (warning only)
        gaps = detect_date_gaps(data_date, master_path)
        if gaps:
            result["warnings"].append(
                f"Missing dates detected: {[str(d) for d in gaps[:5]]}" +
                (f" (and {len(gaps)-5} more)" if len(gaps) > 5 else "")
            )
        result["info"]["date_gaps"] = gaps
        
    except Exception as e:
        result["errors"].append(f"Failed to read/parse file: {str(e)}")
        result["checks_passed"] = False
    
    return result


def display_pre_flight_summary(checks: List[Dict]) -> bool:
    """
    Display pre-flight check summary.
    Returns True if all checks passed.
    """
    print("\n" + "=" * 70)
    print("PRE-FLIGHT CHECK SUMMARY")
    print("=" * 70)
    
    all_passed = True
    
    for check in checks:
        filepath = check["filepath"]
        file_type = check["file_type"].upper()
        
        status = "PASS" if check["checks_passed"] else "FAIL"
        status_symbol = "\u2713" if check["checks_passed"] else "\u2717"
        
        print(f"\n[{status_symbol}] {filepath.name} ({file_type})")
        
        if "data_date" in check["info"]:
            print(f"    Date: {check['info']['data_date']}")
            print(f"    Rows: {check['info']['row_count']}")
            print(f"    Depots: {check['info']['depots']}")
            print(f"    Already processed: {'YES' if check['info'].get('already_processed') else 'NO'}")
        
        if check["errors"]:
            all_passed = False
            print("    ERRORS:")
            for err in check["errors"]:
                print(f"      - {err}")
        
        if check["warnings"]:
            print("    WARNINGS:")
            for warn in check["warnings"]:
                print(f"      - {warn}")
    
    print("\n" + "-" * 70)
    if all_passed:
        print("All pre-flight checks PASSED. Ready to process.")
    else:
        print("Some pre-flight checks FAILED. Fix errors before processing.")
    print("-" * 70)
    
    return all_passed


print("Pre-flight check functions defined")

Pre-flight check functions defined


## 12. Main Pipeline Functions

In [17]:
# =============================================================================
# PROCESS SINGLE FILE
# =============================================================================

def process_depot_file(
    filepath: Path,
    telugu_cal: pd.DataFrame,
    holiday_long: pd.DataFrame,
    update_predictions: bool = True,
    archive: bool = True,
) -> Dict:
    """Process a single depot-level inbound file."""
    result = {"success": False, "file": filepath.name}
    
    try:
        # Read and validate
        df = read_inbound_csv(filepath)
        df = df.loc[:, [c for c in df.columns if c in INBOUND_COLUMNS]]
        df = coerce_depot_types(df)
        
        data_date = df["date"].iloc[0].date()
        result["data_date"] = data_date
        
        # Load and upsert RAW master
        master = load_depot_raw_master(RAW_MASTER_CSV)
        updated_master, inserted, corrected = upsert_depot_raw_master(master, df)
        atomic_write_csv(updated_master, RAW_MASTER_CSV)
        
        result["raw_inserted"] = inserted
        result["raw_corrected"] = corrected
        
        # Build and save GOLD
        gold = build_depot_gold(updated_master, telugu_cal, holiday_long)
        atomic_write_parquet(gold, GOLD_MASTER_PARQ)
        
        result["gold_rows"] = len(gold)
        
        # Update predictions
        predictions_updated = 0
        if update_predictions and PREDICTIONS_FILE.exists():
            pred_df = load_predictions_file(PREDICTIONS_FILE)
            pred_df, predictions_updated = update_predictions_with_actuals(pred_df, gold)
            save_predictions_file(pred_df, PREDICTIONS_FILE)
        
        result["predictions_updated"] = predictions_updated
        
        # Archive
        if archive:
            archived_path = archive_inbound_file(filepath, data_date)
            result["archived_to"] = str(archived_path)
        
        # Log success
        log_record = {
            "ingest_timestamp": datetime.now().isoformat(timespec="seconds"),
            "file_type": "depot",
            "inbound_file": str(filepath.name),
            "inbound_sha256": file_sha256(filepath) if filepath.exists() else "archived",
            "data_date": str(data_date),
            "rows_inserted": inserted,
            "rows_corrected": corrected,
            "gold_rows": len(gold),
            "predictions_updated": predictions_updated,
        }
        log_success(log_record)
        
        result["success"] = True
        
    except Exception as e:
        result["error"] = str(e)
        log_error(filepath, "PROCESSING_ERROR", str(e), "process_depot_file")
    
    return result


def process_service_file(
    filepath: Path,
    telugu_cal: pd.DataFrame,
    holiday_long: pd.DataFrame,
    archive: bool = True,
) -> Dict:
    """Process a single service-level inbound file."""
    result = {"success": False, "file": filepath.name}
    
    try:
        # Read and validate
        df = read_inbound_csv(filepath)
        df = df.loc[:, [c for c in df.columns if c in SERVICE_INBOUND_COLUMNS]]
        df = coerce_service_types(df)
        
        data_date = df["date"].iloc[0].date()
        result["data_date"] = data_date
        
        # Load and upsert RAW master
        master = load_service_raw_master(RAW_SERVICE_MASTER_CSV)
        updated_master, inserted, corrected = upsert_service_raw_master(master, df)
        atomic_write_csv(updated_master, RAW_SERVICE_MASTER_CSV)
        
        result["raw_inserted"] = inserted
        result["raw_corrected"] = corrected
        
        # Build and save GOLD
        gold = build_service_gold(updated_master, telugu_cal, holiday_long)
        atomic_write_parquet(gold, GOLD_SERVICE_PARQ)
        
        result["gold_rows"] = len(gold)
        
        # Archive
        if archive:
            archived_path = archive_inbound_file(filepath, data_date)
            result["archived_to"] = str(archived_path)
        
        # Log success
        log_record = {
            "ingest_timestamp": datetime.now().isoformat(timespec="seconds"),
            "file_type": "service",
            "inbound_file": str(filepath.name),
            "inbound_sha256": file_sha256(filepath) if filepath.exists() else "archived",
            "data_date": str(data_date),
            "rows_inserted": inserted,
            "rows_corrected": corrected,
            "gold_rows": len(gold),
            "predictions_updated": 0,
        }
        log_success(log_record)
        
        result["success"] = True
        
    except Exception as e:
        result["error"] = str(e)
        log_error(filepath, "PROCESSING_ERROR", str(e), "process_service_file")
    
    return result


print("Single file processing functions defined")

Single file processing functions defined


In [18]:
# =============================================================================
# MAIN PIPELINE - PROCESS ALL INBOUND FILES
# =============================================================================

def run_daily_pipeline(
    skip_preflight: bool = False,
    archive_files: bool = True,
    update_predictions: bool = True,
) -> Dict:
    """
    Run the complete daily data pipeline.
    
    Steps:
    1. Scan inbound directory
    2. Run pre-flight checks on all files
    3. Process files in date order (oldest first)
    4. Update RAW masters, GOLD parquets, predictions
    5. Archive processed files
    
    Returns:
        Dict with processing results
    """
    print("\n" + "#" * 70)
    print("#  DAILY DATA PIPELINE")
    print("#" * 70)
    print(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    results = {
        "depot_files_processed": 0,
        "service_files_processed": 0,
        "errors": [],
        "details": [],
    }
    
    # Step 1: Scan inbound directory
    print("\n[Step 1] Scanning inbound directory...")
    files = scan_inbound_files()
    
    depot_files = files["depot"]
    service_files = files["service"]
    unknown_files = files["unknown"]
    
    print(f"  Depot files: {len(depot_files)}")
    print(f"  Service files: {len(service_files)}")
    print(f"  Unknown files: {len(unknown_files)}")
    
    if unknown_files:
        print(f"  WARNING: Unrecognized files will be skipped: {[f.name for f in unknown_files]}")
    
    if not depot_files and not service_files:
        print("\nNo valid inbound files found. Nothing to process.")
        return results
    
    # Step 2: Pre-flight checks
    if not skip_preflight:
        print("\n[Step 2] Running pre-flight checks...")
        
        all_checks = []
        for f in depot_files:
            all_checks.append(pre_flight_check(f, "depot"))
        for f in service_files:
            all_checks.append(pre_flight_check(f, "service"))
        
        all_passed = display_pre_flight_summary(all_checks)
        
        if not all_passed:
            print("\nAborting pipeline due to pre-flight check failures.")
            print("Fix the errors and run again.")
            results["errors"].append("Pre-flight checks failed")
            return results
    
    # Step 3: Load calendars
    print("\n[Step 3] Loading calendars...")
    try:
        telugu_cal = load_telugu_calendar(TELUGU_CAL_PATH)
        holiday_long = load_holiday_calendar_long(HOLIDAY_CAL_PATH)
        print(f"  Telugu calendar: {len(telugu_cal)} rows")
        print(f"  Holiday calendar: {len(holiday_long)} rows")
    except Exception as e:
        print(f"  ERROR loading calendars: {e}")
        results["errors"].append(f"Calendar loading failed: {e}")
        return results
    
    # Step 4: Process depot files (sorted by date)
    if depot_files:
        print("\n[Step 4] Processing depot-level files...")
        
        # Sort by date extracted from filename
        depot_files_sorted = sorted(
            depot_files,
            key=lambda f: extract_date_from_filename(f.name, "depot") or date.max
        )
        
        for f in depot_files_sorted:
            print(f"\n  Processing: {f.name}")
            result = process_depot_file(
                f, telugu_cal, holiday_long,
                update_predictions=update_predictions,
                archive=archive_files
            )
            results["details"].append(result)
            
            if result["success"]:
                results["depot_files_processed"] += 1
                print(f"    Date: {result['data_date']}")
                print(f"    Inserted: {result['raw_inserted']}, Corrected: {result['raw_corrected']}")
                print(f"    Predictions updated: {result.get('predictions_updated', 0)}")
            else:
                print(f"    FAILED: {result.get('error', 'Unknown error')}")
                results["errors"].append(f"{f.name}: {result.get('error')}")
    
    # Step 5: Process service files (sorted by date)
    if service_files:
        print("\n[Step 5] Processing service-level files...")
        
        service_files_sorted = sorted(
            service_files,
            key=lambda f: extract_date_from_filename(f.name, "service") or date.max
        )
        
        for f in service_files_sorted:
            print(f"\n  Processing: {f.name}")
            result = process_service_file(
                f, telugu_cal, holiday_long,
                archive=archive_files
            )
            results["details"].append(result)
            
            if result["success"]:
                results["service_files_processed"] += 1
                print(f"    Date: {result['data_date']}")
                print(f"    Inserted: {result['raw_inserted']}, Corrected: {result['raw_corrected']}")
            else:
                print(f"    FAILED: {result.get('error', 'Unknown error')}")
                results["errors"].append(f"{f.name}: {result.get('error')}")
    
    # Summary
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE")
    print("=" * 70)
    print(f"Depot files processed: {results['depot_files_processed']}")
    print(f"Service files processed: {results['service_files_processed']}")
    print(f"Errors: {len(results['errors'])}")
    
    if results["errors"]:
        print("\nErrors encountered:")
        for err in results["errors"]:
            print(f"  - {err}")
    
    return results


print("Main pipeline function defined:")
print("  - run_daily_pipeline()")

Main pipeline function defined:
  - run_daily_pipeline()


## 13. Execute Pipeline

In [19]:
# =============================================================================
# SCAN INBOUND FILES
# =============================================================================

files = scan_inbound_files()

print("INBOUND FILES DETECTED")
print("=" * 50)

print(f"\nDepot-level files ({len(files['depot'])})")
for f in files["depot"]:
    file_date = extract_date_from_filename(f.name, "depot")
    print(f"  - {f.name} (date: {file_date})")

print(f"\nService-level files ({len(files['service'])})")
for f in files["service"]:
    file_date = extract_date_from_filename(f.name, "service")
    print(f"  - {f.name} (date: {file_date})")

if files["unknown"]:
    print(f"\nUnrecognized files ({len(files['unknown'])})")
    for f in files["unknown"]:
        print(f"  - {f.name} (WILL BE SKIPPED)")

INBOUND FILES DETECTED

Depot-level files (1)
  - ops_daily_2026-01-19.csv (date: 2026-01-19)

Service-level files (1)
  - ops_daily_service_2026-01-19.csv (date: 2026-01-19)


In [20]:
# =============================================================================
# RUN PRE-FLIGHT CHECKS ONLY
# =============================================================================

all_checks = []

for f in files["depot"]:
    all_checks.append(pre_flight_check(f, "depot"))

for f in files["service"]:
    all_checks.append(pre_flight_check(f, "service"))

if all_checks:
    all_passed = display_pre_flight_summary(all_checks)
else:
    print("No files to check.")


PRE-FLIGHT CHECK SUMMARY

[✓] ops_daily_2026-01-19.csv (DEPOT)
    Date: 2026-01-19
    Rows: 4
    Depots: ['CONTONMENT', 'KARIMNAGAR-I', 'NIZAMABAD-I', 'WARANGAL-I']
    Already processed: NO

[✓] ops_daily_service_2026-01-19.csv (SERVICE)
    Date: 2026-01-19
    Rows: 136
    Depots: ['WARANGAL-I']
    Already processed: YES
    WARNINGS:
      - Date 2026-01-19 already exists in master (136 rows). Will be OVERWRITTEN.

----------------------------------------------------------------------
All pre-flight checks PASSED. Ready to process.
----------------------------------------------------------------------


In [21]:
# =============================================================================
# RUN FULL PIPELINE
# =============================================================================

# Uncomment the line below to run the pipeline
results = run_daily_pipeline(skip_preflight=False, archive_files=True, update_predictions=True)


######################################################################
#  DAILY DATA PIPELINE
######################################################################
Run time: 2026-01-29 11:20:10

[Step 1] Scanning inbound directory...
  Depot files: 1
  Service files: 1
  Unknown files: 0

[Step 2] Running pre-flight checks...

PRE-FLIGHT CHECK SUMMARY

[✓] ops_daily_2026-01-19.csv (DEPOT)
    Date: 2026-01-19
    Rows: 4
    Depots: ['CONTONMENT', 'KARIMNAGAR-I', 'NIZAMABAD-I', 'WARANGAL-I']
    Already processed: NO

[✓] ops_daily_service_2026-01-19.csv (SERVICE)
    Date: 2026-01-19
    Rows: 136
    Depots: ['WARANGAL-I']
    Already processed: YES
    WARNINGS:
      - Date 2026-01-19 already exists in master (136 rows). Will be OVERWRITTEN.

----------------------------------------------------------------------
All pre-flight checks PASSED. Ready to process.
----------------------------------------------------------------------

[Step 3] Loading calendars...
  Telugu calendar: 1

## 14. View Data Status

In [22]:
# =============================================================================
# VIEW RAW MASTER STATUS
# =============================================================================

print("RAW MASTER STATUS")
print("=" * 60)

# Depot-level
if RAW_MASTER_CSV.exists():
    df = load_depot_raw_master(RAW_MASTER_CSV)
    print(f"\nDepot-level (ops_daily_master.csv):")
    print(f"  Rows: {len(df)}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Depots: {df['depot'].unique().tolist()}")
else:
    print("\nDepot-level: NOT FOUND")

# Service-level
if RAW_SERVICE_MASTER_CSV.exists():
    df = load_service_raw_master(RAW_SERVICE_MASTER_CSV)
    print(f"\nService-level (ops_daily_service_master.csv):")
    print(f"  Rows: {len(df)}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Unique services: {df['service_number'].nunique()}")
else:
    print("\nService-level: NOT FOUND")

RAW MASTER STATUS

Depot-level (ops_daily_master.csv):
  Rows: 4460
  Date range: 2023-01-01 to 2026-12-01
  Depots: ['CONTONMENT', 'KARIMNAGAR-I', 'NIZAMABAD-I', 'WARANGAL-I']

Service-level (ops_daily_service_master.csv):
  Rows: 2720
  Date range: 2026-01-01 to 2026-12-01
  Unique services: 136


In [23]:
# =============================================================================
# VIEW GOLD PARQUET STATUS
# =============================================================================

print("GOLD PARQUET STATUS")
print("=" * 60)

# Depot-level
if GOLD_MASTER_PARQ.exists():
    df = pd.read_parquet(GOLD_MASTER_PARQ)
    print(f"\nDepot-level (ops_daily_gold.parquet):")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Holidays in data: {df['is_fes_hol'].sum()}")
else:
    print("\nDepot-level GOLD: NOT FOUND")

# Service-level
if GOLD_SERVICE_PARQ.exists():
    df = pd.read_parquet(GOLD_SERVICE_PARQ)
    print(f"\nService-level (ops_daily_service_gold.parquet):")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Unique services: {df['service_number'].nunique()}")
else:
    print("\nService-level GOLD: NOT FOUND")

GOLD PARQUET STATUS

Depot-level (ops_daily_gold.parquet):
  Rows: 4460
  Columns: 14
  Date range: 2023-01-01 to 2026-01-19
  Holidays in data: 504

Service-level (ops_daily_service_gold.parquet):
  Rows: 2720
  Columns: 17
  Date range: 2026-01-01 to 2026-01-20
  Unique services: 136


In [24]:
# =============================================================================
# VIEW LOGS
# =============================================================================

print("INGESTION LOGS")
print("=" * 60)

# Success log
if INGEST_LOG_CSV.exists():
    log_df = pd.read_csv(INGEST_LOG_CSV)
    print(f"\nSuccess log (ingest_log.csv):")
    print(f"  Total ingestions: {len(log_df)}")
    print(f"  Recent entries:")
    print(log_df.tail(5).to_string(index=False))
else:
    print("\nSuccess log: No entries yet")

# Error log
if ERROR_LOG_CSV.exists():
    err_df = pd.read_csv(ERROR_LOG_CSV)
    print(f"\nError log (ingest_errors.csv):")
    print(f"  Total errors: {len(err_df)}")
    if len(err_df) > 0:
        print(f"  Recent errors:")
        print(err_df.tail(5).to_string(index=False))
else:
    print("\nError log: No errors recorded")

INGESTION LOGS

Success log (ingest_log.csv):
  Total ingestions: 10
  Recent entries:
   ingest_timestamp                                                                               inbound_file                                                   inbound_sha256 inbound_date  rows_in_inbound  inserted_rows  corrected_rows  raw_master_rows_after  gold_rows_after file_type  data_date  rows_inserted  rows_corrected  gold_rows  predictions_updated
2026-01-28T13:56:28 /home/sr/agnigent/ai_agents/dynamic_scheduling/data/inbound_daily/ops_daily_2026-01-19.csv d52f68a31d566182d8db139b444c141f7e426cf9fff902a374dcca49410ec464   2026-01-19              4.0            4.0             0.0                 4460.0           4460.0       NaN        NaN            NaN             NaN        NaN                  NaN
2026-01-28T20:47:17 /home/sr/agnigent/ai_agents/dynamic_scheduling/data/inbound_daily/ops_daily_2026-01-19.csv d52f68a31d566182d8db139b444c141f7e426cf9fff902a374dcca49410ec464   2026-01-19   

In [2]:
raw = load_depot_raw_master(RAW_MASTER_CSV)                                         
telugu_cal = load_telugu_calendar(TELUGU_CAL_PATH)                                  
holiday_long = load_holiday_calendar_long(HOLIDAY_CAL_PATH)                         
                                                                                    
gold = build_depot_gold(raw, telugu_cal, holiday_long)                              
atomic_write_parquet(gold, GOLD_MASTER_PARQ)                                        
                                                                                    
print(f"GOLD rebuilt: {len(gold)} rows, {gold['date'].min().date()} to {gold['date'].max().date()}")

NameError: name 'load_depot_raw_master' is not defined

## 15. Summary

### Pipeline Features

| Feature | Description |
|---------|-------------|
| Dual file support | Handles both depot-level and service-level inbound files |
| Filename validation | Must match `ops_daily_YYYY-MM-DD.csv` or `ops_daily_service_YYYY-MM-DD.csv` |
| Date consistency | Filename date must match data date inside file |
| Future date prevention | Rejects files with dates > today |
| Duplicate detection | Warns if date already exists (will overwrite) |
| Gap detection | Warns if there are missing dates |
| Atomic writes | Prevents file corruption during writes |
| Error logging | All errors logged to `ingest_errors.csv` |
| Chronological processing | Files processed oldest-first |

### Daily Workflow

```
1. Place files in data/inbound_daily/
   - ops_daily_YYYY-MM-DD.csv (depot-level)
   - ops_daily_service_YYYY-MM-DD.csv (service-level)

2. Run pre-flight checks (automatic)
   - Validates all files before processing
   - Stops if any critical errors found

3. Processing (if checks pass)
   - Updates RAW masters
   - Rebuilds GOLD parquets
   - Updates predictions with actuals
   - Archives processed files

4. Review logs
   - Check ingest_log.csv for success
   - Check ingest_errors.csv for failures
```

In [3]:

# Run all prerequisite cells first, then rebuild GOLD
%run -i /dev/null

# --- Imports ---
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import re
from datetime import datetime, date, timedelta
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
PROJECT_ROOT = Path("/home/sr/agnigent/ai_agents/dynamic_scheduling")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_MASTER_CSV = RAW_DIR / "ops_daily_master.csv"
GOLD_MASTER_PARQ = PROCESSED_DIR / "ops_daily_gold.parquet"
TELUGU_CAL_PATH = DATA_DIR / "master" / "telugu_calendar.csv"
HOLIDAY_CAL_PATH = DATA_DIR / "master" / "holiday_calendar.csv"

# --- Schemas ---
RAW_COLUMNS = ["depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio"]
TELUGU_CAL_COLUMNS = ["date", "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day"]
HOLIDAY_WIDE_COLUMNS = ["fes_hol_code", "Holiday_Festival", "fes_hol_category",
                        "2023_dates", "2024_dates", "2025_dates", "2026_dates"]
GOLD_COLUMNS = [
    "depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio",
    "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day",
    "fes_hol_code", "Holiday_Festival", "fes_hol_category", "is_fes_hol"
]


Exception: File `'/dev/null'` not found.

In [4]:

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import re
from datetime import datetime, date, timedelta
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
PROJECT_ROOT = Path("/home/sr/agnigent/ai_agents/dynamic_scheduling")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_MASTER_CSV = RAW_DIR / "ops_daily_master.csv"
GOLD_MASTER_PARQ = PROCESSED_DIR / "ops_daily_gold.parquet"
TELUGU_CAL_PATH = DATA_DIR / "master" / "telugu_calendar.csv"
HOLIDAY_CAL_PATH = DATA_DIR / "master" / "holiday_calendar.csv"

RAW_COLUMNS = ["depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio"]
TELUGU_CAL_COLUMNS = ["date", "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day"]
HOLIDAY_WIDE_COLUMNS = ["fes_hol_code", "Holiday_Festival", "fes_hol_category",
                        "2023_dates", "2024_dates", "2025_dates", "2026_dates"]
GOLD_COLUMNS = [
    "depot", "date", "passengers_per_day", "actual_kms", "occupancy_ratio",
    "telugu_thithi", "telugu_paksham", "marriage_day", "telugu_month", "moudyami_day",
    "fes_hol_code", "Holiday_Festival", "fes_hol_category", "is_fes_hol"
]

print("Setup done")


Setup done


In [5]:

# --- Define the functions we need ---

def load_depot_raw_master(path):
    if not path.exists():
        return pd.DataFrame(columns=RAW_COLUMNS)
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df["depot"] = df["depot"].astype("string").str.strip()
    for col in ["passengers_per_day", "actual_kms", "occupancy_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[RAW_COLUMNS].sort_values(["depot", "date"]).reset_index(drop=True)
    return df

def load_telugu_calendar(path):
    cal = pd.read_csv(path)
    cal = cal.loc[:, ~cal.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    cal["date"] = cal["date"].astype(str).str.strip()
    cal["date"] = cal["date"].str.replace(r"\s+00:00:00$", "", regex=True)
    cal["date"] = pd.to_datetime(cal["date"], format="mixed", dayfirst=True, errors="coerce").dt.normalize()
    cal["telugu_thithi"] = cal["telugu_thithi"].astype("string").str.strip()
    cal["telugu_paksham"] = cal["telugu_paksham"].astype("string").str.strip()
    cal["telugu_month"] = cal["telugu_month"].astype("string").str.strip()
    cal["marriage_day"] = pd.to_numeric(cal["marriage_day"], errors="coerce").fillna(0.0)
    cal["moudyami_day"] = pd.to_numeric(cal["moudyami_day"], errors="coerce").fillna(0.0)
    cal = cal[TELUGU_CAL_COLUMNS].sort_values("date").drop_duplicates(subset=["date"])
    return cal

def _split_date_cell(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    s = s.replace("\n", ",").replace(";", ",").replace("|", ",")
    return [p.strip() for p in s.split(",") if p.strip()]

def load_holiday_calendar_long(path):
    hol = pd.read_csv(path)
    hol = hol.loc[:, ~hol.columns.astype(str).str.contains(r"^Unnamed", regex=True)]
    year_cols = ["2023_dates", "2024_dates", "2025_dates", "2026_dates"]
    long = hol.melt(
        id_vars=["fes_hol_code", "Holiday_Festival", "fes_hol_category"],
        value_vars=year_cols, var_name="year_col", value_name="date_list",
    )
    long["date_strs"] = long["date_list"].apply(_split_date_cell)
    long = long.explode("date_strs", ignore_index=True)
    long["date"] = pd.to_datetime(long["date_strs"], dayfirst=True, errors="coerce")
    long = long.dropna(subset=["date"])
    long = long[["date", "fes_hol_code", "Holiday_Festival", "fes_hol_category"]].copy()
    long["fes_hol_code"] = pd.to_numeric(long["fes_hol_code"], errors="coerce").astype("Int64")
    long = long.sort_values(["date", "fes_hol_code"]).drop_duplicates(subset=["date"], keep="first").reset_index(drop=True)
    return long

def build_depot_gold(raw_df, telugu_cal_df, holiday_long_df):
    df = raw_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date_key"] = df["date"].dt.normalize()
    df["depot"] = df["depot"].astype("string")
    tel = telugu_cal_df.copy()
    tel["date"] = pd.to_datetime(tel["date"], errors="coerce")
    tel["date_key"] = tel["date"].dt.normalize()
    tel = tel.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    hol = holiday_long_df.copy()
    hol["date"] = pd.to_datetime(hol["date"], errors="coerce")
    hol["date_key"] = hol["date"].dt.normalize()
    hol = hol.drop(columns=["date"]).drop_duplicates(subset=["date_key"])
    df = df.merge(tel, on="date_key", how="left")
    df = df.merge(hol, on="date_key", how="left")
    df["is_fes_hol"] = df["fes_hol_code"].notna()
    df["fes_hol_code"] = df["fes_hol_code"].fillna(0).astype("Int64")
    df["Holiday_Festival"] = df["Holiday_Festival"].fillna("NONE").astype("string")
    df["fes_hol_category"] = df["fes_hol_category"].fillna("NONE").astype("string")
    df["is_fes_hol"] = df["is_fes_hol"].astype("int8")
    df = df.drop(columns=["date_key"])
    df = df[GOLD_COLUMNS].sort_values(["depot", "date"]).reset_index(drop=True)
    return df

def atomic_write_parquet(df, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, engine="pyarrow", index=False)
    tmp.replace(path)

print("Functions defined")


Functions defined


In [6]:

# Rebuild GOLD from current RAW master
raw = load_depot_raw_master(RAW_MASTER_CSV)
telugu_cal = load_telugu_calendar(TELUGU_CAL_PATH)
holiday_long = load_holiday_calendar_long(HOLIDAY_CAL_PATH)

gold = build_depot_gold(raw, telugu_cal, holiday_long)
atomic_write_parquet(gold, GOLD_MASTER_PARQ)

print(f"GOLD rebuilt: {len(gold)} rows, {gold['date'].min().date()} to {gold['date'].max().date()}")


# ---- Update pending predictions with actuals from fresh GOLD ----
PREDICTIONS_FILE = Path("/home/sr/agnigent/ai_agents/dynamic_scheduling/output/predictions/daily_predictions.parquet")

if PREDICTIONS_FILE.exists():
    pred_df = pd.read_parquet(PREDICTIONS_FILE)
    pending_mask = pred_df['status'] == 'pending'
    pending_count = pending_mask.sum()

    if pending_count > 0:
        gold_lookup = gold.copy()
        gold_lookup['date'] = pd.to_datetime(gold_lookup['date'])
        pred_df['prediction_date'] = pd.to_datetime(pred_df['prediction_date'])

        # Aggregate gold to depot-day level
        agg_cols = {'passengers_per_day': 'sum', 'occupancy_ratio': 'mean'}
        if 'actual_kms' in gold_lookup.columns:
            agg_cols['actual_kms'] = 'sum'
        actuals = gold_lookup.groupby(['depot', 'date']).agg(agg_cols).reset_index()

        updates_count = 0
        for idx in pred_df[pending_mask].index:
            row = pred_df.loc[idx]
            match = actuals[
                (actuals['date'] == row['prediction_date']) &
                (actuals['depot'] == row['depot'])
            ]
            if len(match) == 0:
                continue

            actual_passengers = match['passengers_per_day'].values[0]
            actual_or = match['occupancy_ratio'].values[0] if 'occupancy_ratio' in match.columns else None
            actual_kms_val = match['actual_kms'].values[0] if 'actual_kms' in match.columns else None
            predicted = row['predicted_passengers']
            estimated_kms = row['estimated_kms']

            passenger_error = predicted - actual_passengers if predicted else None
            passenger_error_pct = (passenger_error / actual_passengers * 100) if actual_passengers and actual_passengers > 0 else None

            km_error = None
            km_error_pct = None
            if actual_kms_val and actual_kms_val > 0 and estimated_kms:
                km_error = estimated_kms - actual_kms_val
                km_error_pct = (km_error / actual_kms_val * 100)

            pred_df.loc[idx, 'actual_passengers'] = actual_passengers
            pred_df.loc[idx, 'actual_or'] = actual_or
            pred_df.loc[idx, 'actual_kms'] = actual_kms_val
            pred_df.loc[idx, 'passenger_error'] = passenger_error
            pred_df.loc[idx, 'passenger_error_pct'] = passenger_error_pct
            pred_df.loc[idx, 'km_error'] = km_error
            pred_df.loc[idx, 'km_error_pct'] = km_error_pct
            pred_df.loc[idx, 'status'] = 'completed'
            updates_count += 1

        if updates_count > 0:
            for col in ['run_date', 'prediction_date']:
                if col in pred_df.columns:
                    pred_df[col] = pd.to_datetime(pred_df[col])
            pred_df.to_parquet(PREDICTIONS_FILE, index=False)
            print(f'Updated {updates_count} pending predictions with actuals')
        else:
            print(f'No pending predictions matched GOLD dates ({pending_count} still pending)')
    else:
        print('No pending predictions to update')
else:
    print('No predictions file found - skipping actuals update')


GOLD rebuilt: 4460 rows, 2023-01-01 to 2026-12-01
